# Sesión en vivo — inferencia causal

Ejecuta las celdas en orden durante la clase. Cada módulo se lee de
`data/module_a.csv`, `data/module_b.csv`, `data/module_c.csv` (exportados
desde las Hojas de cálculo de los 3 Google Forms — ver
`docs/GOOGLE_FORMS_SETUP.md`).

Si esos archivos todavía no existen, `SIMULATE = True` genera datos
**ficticios** solo para probar que el código corre antes de la clase real
— nunca los confundas con resultados reales, se marcan explícitamente.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path("data")
SIMULATE = not (DATA_DIR / "module_a.csv").exists()

pd.set_option("display.width", 120)
rng = np.random.default_rng(7)

EDAD_MID = {"<20": 18, "20-24": 22, "25-29": 27, "30-39": 34.5, "40+": 45}

if SIMULATE:
    print("Usando datos SIMULADOS de prueba -- no son respuestas reales.")


In [ ]:
def simulate_module_a(n=70):
    rut = rng.integers(0, 100, n)
    edad = rng.choice(list(EDAD_MID), n)
    genero = rng.choice(["Mujer", "Hombre", "Otro"], n, p=[.48, .48, .04])
    relig = rng.integers(1, 6, n)
    practica = rng.choice(["Sí", "No"], n)
    # WTP sube ~modestamente con el dígito del RUT (anclaje débil, ruidoso)
    wine = np.clip(6000 + 60 * rut + rng.normal(0, 2500, n), 500, None).round(-2)
    earbuds = np.clip(9000 + 45 * rut + rng.normal(0, 3500, n), 500, None).round(-2)
    return pd.DataFrame({
        "rut_digitos": rut, "wtp_vino": wine, "wtp_audifonos": earbuds,
        "edad_rango": edad, "genero": genero, "religiosidad": relig,
        "practica_religion": practica,
    })

ITEMS_B = [
    ("gandhi", 78, 9, 140), ("einstein", 76, 9, 135), ("neruda", 69, 8, 125),
    ("allende", 65, 7, 115), ("parra", 49, 6, 90),
]

def simulate_module_b(n=70):
    rows = []
    for _ in range(n):
        order = rng.permutation(len(ITEMS_B))
        edad = rng.choice(list(EDAD_MID)); genero = rng.choice(["Mujer", "Hombre", "Otro"], p=[.48, .48, .04])
        relig = rng.integers(1, 6); practica = rng.choice(["Sí", "No"])
        items = []
        for pos, idx in enumerate(order, start=1):
            item, true_val, low, high = ITEMS_B[idx]
            alto = rng.random() < 0.5
            anchor_value = high if alto else low
            # el anclaje empuja la estimacion hacia el ancla, y el jalón decae con la posición
            pull = (0.35 - 0.05 * (pos - 1)) * (anchor_value - true_val)
            est = true_val + pull + rng.normal(0, 8)
            items.append({
                "item": item, "true_value": true_val,
                "anchor_level": "alto" if alto else "bajo", "anchor_value": anchor_value,
                "position": pos, "estimacion": round(float(np.clip(est, 1, 200)), 1),
            })
        rows.append({
            "items_json": json.dumps(items), "edad_rango": edad, "genero": genero,
            "religiosidad": relig, "practica_religion": practica,
        })
    return pd.DataFrame(rows)

TIPI_TRAITS = [
    ("extraversion", False), ("amabilidad", True), ("meticulosidad", False),
    ("estabilidad_emocional", True), ("apertura", False), ("extraversion", True),
    ("amabilidad", False), ("meticulosidad", True), ("estabilidad_emocional", False),
    ("apertura", True),
]

def simulate_module_c(n=70):
    rows = []
    for _ in range(n):
        variante = rng.choice(["switch", "push"])
        genero = rng.choice(["Mujer", "Hombre", "Otro"], p=[.48, .48, .04])
        relig = rng.integers(1, 6); practica = rng.choice(["Sí", "No"])
        base = 5.2 if variante == "switch" else 3.4  # push se juzga menos aceptable
        accept = np.clip(rng.normal(base, 1.3), 1, 7)
        tipi = [{"trait": t, "r": r, "value": int(rng.integers(1, 8))} for t, r in TIPI_TRAITS]
        rows.append({
            "variante": variante, "aceptabilidad": round(float(accept), 1),
            "genero": genero, "religiosidad": relig, "practica_religion": practica,
            "tipi_json": json.dumps(tipi),
        })
    return pd.DataFrame(rows)


## Módulo A — El precio arbitrario

Correlación entre el dígito del RUT y la disposición a pagar (WTP), y
balance de covariables al partir la muestra en la mediana del dígito.

In [ ]:
df_a = simulate_module_a() if SIMULATE else pd.read_csv(DATA_DIR / "module_a.csv")
df_a["edad_num"] = df_a["edad_rango"].map(EDAD_MID)
df_a["mujer"] = (df_a["genero"] == "Mujer").astype(int)
df_a.head()


In [ ]:
# Correlación / regresión: WTP ~ dígito del RUT
for outcome in ["wtp_vino", "wtp_audifonos"]:
    m = smf.ols(f"{outcome} ~ rut_digitos", data=df_a).fit(cov_type="HC1")
    r = np.corrcoef(df_a["rut_digitos"], df_a[outcome])[0, 1]
    print(f"{outcome}: r = {r:.2f}, beta(dígito) = {m.params['rut_digitos']:.1f} "
          f"(p = {m.pvalues['rut_digitos']:.3f})")


In [ ]:
# Gráfico clásico de Ariely: WTP promedio por decil del dígito del RUT
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_a["decil_rut"] = pd.qcut(df_a["rut_digitos"], 10, labels=False, duplicates="drop")
for ax, outcome, title in zip(axes, ["wtp_vino", "wtp_audifonos"], ["Vino", "Audífonos"]):
    df_a.groupby("decil_rut")[outcome].mean().plot(kind="bar", ax=ax, color="#2B6E6B")
    ax.set_title(title); ax.set_xlabel("Decil del dígito del RUT"); ax.set_ylabel("WTP promedio (CLP)")
plt.tight_layout(); plt.show()


In [ ]:
def balance_table(df, split_col, label):
    """Compara covariables entre las dos mitades de `split_col` (bool)."""
    rows = []
    for col, name in [("edad_num", "Edad"), ("mujer", "% mujer"), ("religiosidad", "Religiosidad (1-5)")]:
        g0 = df.loc[~df[split_col], col].astype(float)
        g1 = df.loc[df[split_col], col].astype(float)
        diff = g1.mean() - g0.mean()
        se = np.sqrt(g0.var(ddof=1) / len(g0) + g1.var(ddof=1) / len(g1))
        t = diff / se if se > 0 else np.nan
        rows.append({"Covariable": name, "Grupo 0": round(g0.mean(), 2), "Grupo 1": round(g1.mean(), 2),
                     "Diferencia": round(diff, 2), "t": round(t, 2)})
    out = pd.DataFrame(rows)
    print(f"--- Balance: {label} ---")
    return out

df_a["rut_alto"] = df_a["rut_digitos"] >= df_a["rut_digitos"].median()
balance_a = balance_table(df_a, "rut_alto", "Módulo A -- split por mediana del dígito del RUT (NO es aleatorización del investigador)")
balance_a


## Módulo B — El ancla numérica

Cinco ítems, ancla alta/baja **asignada al azar** por el sitio, orden de
presentación también aleatorio. Primero el ATE global y por posición
(para ver si el anclaje decae con la repetición); luego el mismo balance
de covariables que en el Módulo A, para comparar.

In [ ]:
df_b_wide = simulate_module_b() if SIMULATE else pd.read_csv(DATA_DIR / "module_b.csv")
df_b_wide = df_b_wide.reset_index().rename(columns={"index": "submission_id"})
df_b_wide["edad_num"] = df_b_wide["edad_rango"].map(EDAD_MID)
df_b_wide["mujer"] = (df_b_wide["genero"] == "Mujer").astype(int)

# Expandir items_json a formato largo: 5 filas por respuesta
long_rows = []
for _, row in df_b_wide.iterrows():
    for item in json.loads(row["items_json"]):
        long_rows.append({**item, "submission_id": row["submission_id"]})
df_b = pd.DataFrame(long_rows)
df_b.head()


In [ ]:
# Estandarizar la estimación dentro de cada ítem (para comparar Gandhi con Violeta Parra en la misma escala)
df_b["estimacion_z"] = df_b.groupby("item")["estimacion"].transform(lambda s: (s - s.mean()) / s.std())
df_b["alto"] = (df_b["anchor_level"] == "alto").astype(int)

ate_global = df_b.loc[df_b["alto"] == 1, "estimacion_z"].mean() - df_b.loc[df_b["alto"] == 0, "estimacion_z"].mean()
print(f"ATE global (ancla alta - ancla baja), en desviaciones estándar del ítem: {ate_global:.2f}")

ate_by_pos = df_b.groupby(["position", "anchor_level"])["estimacion_z"].mean().unstack()
ate_by_pos["gap"] = ate_by_pos["alto"] - ate_by_pos["bajo"]
print(ate_by_pos)

fig, ax = plt.subplots(figsize=(6, 4))
ate_by_pos["gap"].plot(marker="o", ax=ax, color="#B9862A")
ax.axhline(0, color="grey", lw=.8)
ax.set_xlabel("Posición en la secuencia (1-5)"); ax.set_ylabel("Brecha de anclaje (alto - bajo, z)")
ax.set_title("¿El efecto de anclaje decae con la repetición?")
plt.tight_layout(); plt.show()


In [ ]:
# Regresión con efectos fijos por ítem y errores estándar agrupados por estudiante
model_b = smf.ols(
    "estimacion_z ~ alto + position + alto:position + C(item)", data=df_b
).fit(cov_type="cluster", cov_kwds={"groups": df_b["submission_id"]})
print(model_b.summary().tables[1])


In [ ]:
df_b_wide["alto_prom"] = df_b.groupby("submission_id")["alto"].mean().reindex(df_b_wide["submission_id"]).values
df_b_wide["mitad_alta"] = df_b_wide["alto_prom"] >= df_b_wide["alto_prom"].median()
balance_b = balance_table(df_b_wide, "mitad_alta", "Módulo B -- asignación aleatoria del ancla (SÍ es aleatorización del investigador)")
balance_b


### A vs. B, lado a lado

Ambos pueden salir balanceados -- ese no es el punto. La diferencia es que
B lo *garantiza por construcción*, incluso en variables que no medimos; A
solo lo sugiere, para las variables que sí medimos.

In [ ]:
pd.concat([balance_a.set_index("Covariable"), balance_b.set_index("Covariable")],
          axis=1, keys=["Módulo A (dígito RUT)", "Módulo B (azar del sitio)"])


## Módulo C — El dilema y tú

Variante del tranvía asignada al azar (switch = palanca, push = empujar).
ATE global sobre la aceptabilidad (1-7), y heterogeneidad por género,
religiosidad y personalidad (TIPI-10).

In [ ]:
df_c = simulate_module_c() if SIMULATE else pd.read_csv(DATA_DIR / "module_c.csv")
df_c["push"] = (df_c["variante"] == "push").astype(int)
df_c["mujer"] = (df_c["genero"] == "Mujer").astype(int)
df_c["religioso"] = (df_c["religiosidad"].astype(float) >= 4).astype(int)

def score_tipi(tipi_json):
    items = json.loads(tipi_json)
    scores = {}
    for it in items:
        v = 8 - it["value"] if it["r"] else it["value"]
        scores.setdefault(it["trait"], []).append(v)
    return {t: np.mean(v) for t, v in scores.items()}

tipi_scores = df_c["tipi_json"].apply(score_tipi).apply(pd.Series)
df_c = pd.concat([df_c, tipi_scores], axis=1)
df_c.head()


In [ ]:
# ATE global: empujar vs. jalar la palanca
ate_c = smf.ols("aceptabilidad ~ push", data=df_c).fit(cov_type="HC1")
print(ate_c.summary().tables[1])


In [ ]:
df_c["consciente_alto"] = (df_c["meticulosidad"] >= df_c["meticulosidad"].median()).astype(int)

subgroups = {
    "Mujer": "mujer",
    "Religioso/a (4-5)": "religioso",
    "Meticulosidad alta": "consciente_alto",
}
rows = []
for label, col in subgroups.items():
    m = smf.ols(f"aceptabilidad ~ push * {col}", data=df_c).fit(cov_type="HC1")
    rows.append({"Subgrupo": label, "ATE (push)": m.params["push"],
                 "Interacción": m.params[f"push:{col}"], "p (interacción)": m.pvalues[f"push:{col}"]})
het_table = pd.DataFrame(rows)
het_table


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(het_table["Interacción"], het_table["Subgrupo"], fmt="o", color="#6E4A76")
ax.axvline(0, color="grey", lw=.8)
ax.set_xlabel("Interacción push × subgrupo (CATE - ATE)")
ax.set_title("Heterogeneidad del efecto por subgrupo")
plt.tight_layout(); plt.show()
